# Nettoyage des données

## 1. Chargement des données

In [1]:
import pandas as pd
import numpy as np
import os

df_evenements = pd.read_csv('../data/raw/Dome_evenements.csv')
df_billets = pd.read_csv('../data/raw/Dome_billets.csv')

print("Événements bruts :", df_evenements.shape)
print("Billets bruts    :", df_billets.shape)

Événements bruts : (98, 8)
Billets bruts    : (820, 12)


## 2. Nettoyage du DataSet evenements

In [2]:
evenements = df_evenements.drop_duplicates().copy()

print("Avant :", len(df_evenements), "→ Après :", len(evenements))

Avant : 98 → Après : 96


Suppression des 2 événements présents en double à l'identique (EV331, EV340) : 98 → 96 événements (audit des événements, section 2).

In [3]:

masque_zero = evenements['billets_vendus'] == 0
evenements.loc[masque_zero, 'billets_vendus'] = np.nan

print("Valeurs passées à NaN :", masque_zero.sum())

Valeurs passées à NaN : 3


Les 3 événements à 0 billet vendu (EV374, EV303, EV355) ont 12, 7 et 6 billets dans l'échantillon : leur 0 est une valeur manquante et non une annulation (audit des billets, étape 7). Leurs ventes sont donc passées à inconnu ; ces événements seront exclus des analyses de remplissage et de prévision (Q1 à Q4).

In [4]:
evenements['billets_vendus'] = evenements['billets_vendus'].astype('Int64')

print(evenements['billets_vendus'].dtype)
print("Valeurs manquantes :", evenements['billets_vendus'].isnull().sum())

Int64
Valeurs manquantes : 3


`billets_vendus` était lue en décimal à cause du format d'export (valeurs écrites avec « .0 ») ; elle est convertie en entier `Int64`, le type de pandas qui accepte les valeurs manquantes.

In [5]:
masque_notoriete_nan = evenements['notoriete_artiste'].isnull()
print("Notoriétés manquantes :", masque_notoriete_nan.sum())
display(evenements.loc[masque_notoriete_nan, ['evenement_id', 'genre', 'notoriete_artiste']])

mediane_par_genre = evenements.groupby('genre')['notoriete_artiste'].transform('median')
evenements['notoriete_artiste'] = evenements['notoriete_artiste'].fillna(mediane_par_genre)

print("Notoriétés manquantes après :", evenements['notoriete_artiste'].isnull().sum())

Notoriétés manquantes : 2


,evenement_id,genre,notoriete_artiste
3,EV362,rap,NaN
55,EV303,rock,NaN


Notoriétés manquantes après : 0


Les 2 notoriétés manquantes (EV362, rap, et EV303, rock) sont imputées par la médiane de notoriété de leur genre : la colonne est asymétrique, la médiane est plus robuste que la moyenne, et le genre donne une estimation plus proche que la médiane globale. EV303 est de toute façon exclu de Q1 à Q4.

In [6]:
evenements['date'] = pd.to_datetime(evenements['date'])

print(evenements['date'].dtype)
print("Du", evenements['date'].min().date(), "au", evenements['date'].max().date())

datetime64[us]
Du 2025-09-01 au 2026-09-16


La date est convertie en type date. Elle sert uniquement à situer l'événement dans la saison : le jour de la semaine retenu pour l'analyse reste `jour_semaine` (audit des événements, section 6).

In [7]:
evenements['taux_remplissage'] = evenements['billets_vendus'] / evenements['capacite']

display(evenements['taux_remplissage'].describe().round(3))
print("Taux manquants :", evenements['taux_remplissage'].isnull().sum())

count     93.0
mean     0.584
std      0.199
min      0.235
25%      0.427
50%      0.558
75%      0.691
max        1.0
Name: taux_remplissage, dtype: Float64

Taux manquants : 3


Ajout du taux de remplissage (billets vendus / capacité), variable centrale de Q1. Sans les 3 valeurs inconnues, il va de 23,5 % à 100 % (moyenne 58 %, médiane 56 %).

In [8]:
print(evenements.shape)
print(evenements.dtypes)
evenements.head()

(96, 9)
evenement_id                    str
genre                           str
jour_semaine                    str
capacite                      int64
prix_moyen_eur              float64
notoriete_artiste           float64
billets_vendus                Int64
date                 datetime64[us]
taux_remplissage            Float64
dtype: object


,evenement_id,genre,jour_semaine,capacite,prix_moyen_eur,notoriete_artiste,billets_vendus,date,taux_remplissage
0,EV320,rap,dim,9800,18.00,26.0,4923,2025-11-20,0.502347
1,EV370,spectacle,mar,15800,74.23,20.0,6051,2026-06-08,0.382975
2,EV377,sport,sam,6500,45.99,53.0,5221,2026-07-06,0.803231
3,EV362,rap,ven,9800,70.96,41.0,6203,2026-05-07,0.632959
4,EV394,sport,dim,9800,46.33,100.0,7196,2026-09-12,0.734286


Fichier des événements nettoyé : 96 événements, 9 colonnes (les 8 d'origine et `taux_remplissage`).

## 3. Nettoyage du DataSet Billets

In [9]:
billets = df_billets.drop_duplicates().copy()

print("Avant :", len(df_billets), "→ Après :", len(billets))

Avant : 820 → Après : 700


Suppression des 120 billets présents en double à l'identique : 820 → 700 billets (audit des billets, étape 1).

In [10]:
masque_age_12 = billets['age'] == 12
print("Billets à 12 ans (après dédoublonnage) :", masque_age_12.sum())

billets.loc[masque_age_12, 'age'] = np.nan

print("Âges inconnus au total :", billets['age'].isnull().sum())

Billets à 12 ans (après dédoublonnage) : 36
Âges inconnus au total : 50


Les 36 billets à exactement 12 ans (après dédoublonnage) sont passés à inconnu : 12 est une valeur plancher ou par défaut, pas un âge réel, comme le montrent l'empilement sur cette valeur et une consommation au bar supérieure à celle des adultes (audit des billets, étape 4). Avec les 14 âges déjà manquants, 50 âges sont inconnus.

In [11]:
billets['age_inconnu'] = billets['age'].isnull()

print(billets['age_inconnu'].value_counts())

age_inconnu
False    650
True      50
Name: count, dtype: int64


L'indicateur `age_inconnu` est créé avant l'imputation, pour garder la trace des 50 âges qui seront estimés : les analyses de profil (Q7) pourront ainsi en tenir compte.

In [12]:
mediane_age = billets['age'].median()
print("Médiane des âges connus :", mediane_age)

billets['age'] = billets['age'].fillna(mediane_age)

print("Âges manquants après :", billets['age'].isnull().sum())

Médiane des âges connus : 35.0
Âges manquants après : 0


Les 50 âges inconnus sont imputés par la médiane des âges connus (35 ans), plus robuste que la moyenne pour une distribution asymétrique.

In [13]:
billets['prix_plancher'] = billets['prix_paye_eur'] == 12

print("Billets au plancher de 12 € :", billets['prix_plancher'].sum())

Billets au plancher de 12 € : 26


26 billets (après dédoublonnage) sont au plancher de 12 € appliqué par le système : leur vraie valeur est ≤ 12 €. Ils sont conservés et signalés par l'indicateur `prix_plancher` (audit des billets, étape 4).

In [14]:
billets['invitation'] = billets['prix_paye_eur'] == 0

print("Invitations (billets à 0 €) :", billets['invitation'].sum())

Invitations (billets à 0 €) : 3


Les 3 billets à 0 € sont des invitations probables : conservés, ils sont signalés par l'indicateur `invitation` pour pouvoir être exclus des analyses de prix.

In [15]:
print("Consommations inconnues (NaN) :", billets['conso_bar_eur'].isnull().sum())
print("Consommations à 0 € :", (billets['conso_bar_eur'] == 0).sum())

Consommations inconnues (NaN) : 14
Consommations à 0 € : 72


`conso_bar_eur` n'est pas modifiée : ses 14 valeurs manquantes restent inconnues (un `NaN` n'est pas un 0), et les 72 billets à 0 € sont de vrais non-consommateurs. Les billets à consommation inconnue seront exclus de la modélisation de la consommation (Q5) au moment de la traiter.

In [16]:
correspondance = {'oui': True, 'non': False}

billets['deja_venu'] = billets['deja_venu'].map(correspondance)
billets['revient_saison_suivante'] = billets['revient_saison_suivante'].map(correspondance)

print(billets[['deja_venu', 'revient_saison_suivante']].dtypes)
print(billets['deja_venu'].value_counts())

deja_venu                  bool
revient_saison_suivante    bool
dtype: object
deja_venu
False    401
True     299
Name: count, dtype: int64


`deja_venu` et `revient_saison_suivante` sont converties de « oui / non » en booléens, plus simples à utiliser dans les calculs et les modèles. `revient_saison_suivante` est conservée, mais ne servira pas de variable explicative pour Q5, car elle est postérieure à l'achat.

In [17]:
print(billets.shape)
print(billets.dtypes)
billets.head()

(700, 15)
billet_id                      str
evenement_id                   str
zone                           str
prix_paye_eur              float64
achat_jours_avant          float64
arrivee_min_avant          float64
x_salle_m                  float64
y_salle_m                  float64
conso_bar_eur              float64
age                        float64
deja_venu                     bool
revient_saison_suivante       bool
age_inconnu                   bool
prix_plancher                 bool
invitation                    bool
dtype: object


,billet_id,evenement_id,zone,prix_paye_eur,achat_jours_avant,arrivee_min_avant,x_salle_m,y_salle_m,conso_bar_eur,age,deja_venu,revient_saison_suivante,age_inconnu,prix_plancher,invitation
0,BL600680,EV369,loge,73.66,53.0,23.0,16.6,-13.5,28.63,41.0,False,True,False,False,False
1,BL600471,EV386,fosse,122.64,44.0,56.0,0.7,-1.7,10.63,44.0,False,False,False,False,False
2,BL600421,EV313,gradin bas,59.81,29.0,34.0,-10.3,-21.7,3.01,32.0,False,False,False,False,False
3,BL600643,EV342,gradin bas,86.22,33.0,32.0,12.9,21.4,4.71,42.0,False,False,False,False,False
4,BL600447,EV388,gradin haut,67.45,102.0,81.0,-35.3,42.9,NaN,30.0,True,True,False,False,False


Fichier des billets nettoyé : 700 billets, 15 colonnes (les 12 d'origine et les indicateurs `age_inconnu`, `prix_plancher` et `invitation`).

## 4. Vérifications

In [18]:
assert len(evenements) == 96 , "Nombre d'événements inattendu"
assert evenements['evenement_id'].is_unique, "Identifiant d'événement en double"
assert evenements['billets_vendus'].isnull().sum() == 3 , "Il devrait rester 3 ventes inconnues"
assert evenements['notoriete_artiste'].isnull().sum() == 0 , "Notoriété encore manquante"
assert (evenements['taux_remplissage'].dropna() <= 1).all(), "Taux de remplissage supérieur à 100 %"

print("Vérifications des événements : OK")

Vérifications des événements : OK


In [19]:
assert len(billets) == 700, "Nombre de billets inattendu"
assert billets['billet_id'].is_unique, "Identifiant de billet en double"
assert billets['age'].isnull().sum() == 0, "Âge encore manquant"
assert (billets['age'] == 12).sum() == 0, "Il reste des âges à 12 ans"
assert billets['age'].dtype == 'float64', "La colonne age n'est pas numérique"
assert billets['age_inconnu'].sum() == 50, "Nombre d'âges imputés inattendu"
assert billets['conso_bar_eur'].isnull().sum() == 14, "La consommation ne doit pas être imputée"
assert billets['evenement_id'].isin(evenements['evenement_id']).all(), "Billet pointant vers un événement absent"

print("Vérifications des billets : OK")

Vérifications des billets : OK


Ces contrôles font échouer le notebook si une étape du nettoyage ne produit pas le résultat attendu : toute modification future qui casserait une décision sera détectée immédiatement.

## 5. Enregistrement des fichiers nettoyés

In [20]:

os.makedirs('../data/processed', exist_ok= True)

evenements.to_csv('../data/processed/evenements_clean.csv', index= False)
billets.to_csv('../data/processed/billets_clean.csv', index= False)

print("Fichiers enregistrés :", os.listdir('../data/processed'))

Fichiers enregistrés : ['billets_clean.csv', 'evenements_clean.csv']


Les fichiers nettoyés sont enregistrés dans `data/processed/` ; tous les notebooks d'analyse partent de ces deux fichiers. À la relecture, la date doit être reconvertie (`parse_dates=['date']`), car le format CSV ne conserve pas les types.